# 1 · Calibration walkthrough

Estimate a camera's intrinsics and lens distortion from the bundled checkerboard images, then read the diagnostics that say whether to trust the result. The sample images are synthetic, with exact ground truth in `data/ground_truth.json`.

In [1]:
import os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np, cv2

## Collect the board views

A `CheckerboardSpec` fixes the inner-corner grid and square size; the calibrator detects and sub-pixel-refines the corners in every image.

In [2]:
from campose import CameraCalibrator, CheckerboardSpec

spec = CheckerboardSpec(9, 6, square_size=0.025)
calibrator = CameraCalibrator(spec)
found = calibrator.add_images('data/sample_calibration_images')
print(f'detected the board in {found} images')

detected the board in 21 images


## Solve and summarise

`calibrate` returns `K`, distortion, per-image board poses, and the reprojection error, with a plain-language verdict.

In [3]:
result = calibrator.calibrate()
print(result.summary())

Calibration summary
-------------------
images used        : 21
image size (w x h) : 640 x 480
overall RMS error  : 0.0735 px

Intrinsic matrix K
  fx, fy : 919.35, 917.40  (focal length in pixels)
  cx, cy : 327.53, 244.80  (principal point)

Distortion coefficients
  radial     k1, k2, k3 : -0.28696, +0.57690, -2.28356
  tangential p1, p2     : +0.00117, -0.00052

worst image        : board_11.png at 0.1513 px

verdict            : good (RMS < 0.5 px)


## Per-image error and outliers

One bad frame drags the whole fit down. The bar chart marks anything above mean + 2σ in red.

In [4]:
from campose import visualization as viz
viz.plot_reprojection_errors(result)
plt.show()
print('flagged outliers:', [p.source for p in calibrator.flag_outliers(result)])

flagged outliers: ['board_11.png', 'board_19.png']


## What the lens is doing

The displacement map shows how far the distortion model pushes each pixel.

In [5]:
viz.plot_distortion_map(result.intrinsics, result.image_size)
plt.show()

## Where the boards sat

The recovered extrinsics place every board in 3D relative to the camera at the origin.

In [6]:
viz.plot_board_poses_3d(result, spec)
plt.show()

## Is the calibration stable?

Bootstrap resampling recalibrates on random subsets; a small standard deviation per parameter means enough good images.

In [7]:
confidence = calibrator.bootstrap_confidence(trials=30)
for name, value in confidence['std'].items():
    print(f'{name:>3}: std {value:.4f}')

 fx: std 0.5364
 fy: std 0.5282
 cx: std 0.4961
 cy: std 0.2558
 k1: std 0.0043
 k2: std 0.1123
 p1: std 0.0001
 p2: std 0.0001
 k3: std 1.0195


## Check against ground truth

Because the data is synthetic we can compare the estimate to the exact camera that produced it.

In [8]:
import json
truth = json.loads(Path('data/ground_truth.json').read_text())
K_true = np.array(truth['camera_matrix'])
print('true fx, fy, cx, cy :', K_true[0,0], K_true[1,1], K_true[0,2], K_true[1,2])
k = result.intrinsics
print('est  fx, fy, cx, cy :', round(k.fx,2), round(k.fy,2), round(k.cx,2), round(k.cy,2))

true fx, fy, cx, cy : 920.0 918.0 328.0 244.0
est  fx, fy, cx, cy : 919.35 917.4 327.53 244.8


Save it for reuse in the pose notebook:

In [9]:
from campose import io
io.save(result, 'data/calibration_results/sample_calibration.json')
print('saved')

saved
